# Batch Size One Breaks Batch Norm

**Topic:** Layer norm vs batch norm &nbsp;·&nbsp; **Difficulty:** Easy

Normalize across the batch and a batch of one has zero variance: every feature
becomes exactly `(x - x) / sqrt(0 + eps) = 0`, and the layer destroys its input
entirely. This is not a corner case, and it is why sequence models with variable
batch sizes use layer norm.

Layer norm normalizes across the *features* of each example independently, so it
does not care how many examples you hand it. Implement both and watch one of them
fail.

## The data (created in the next cell)

- `X` `(32, 128)`: a batch of 32 examples with 128 features each.
- `eps = 1e-5`.

## Your task

With **NumPy only** (`gamma = 1`, `beta = 0` throughout):

1. `batchnorm(x)`: normalize each **feature** using that feature's mean and
   variance across the batch dimension.
2. `layernorm(x)`: normalize each **example** using the mean and variance across
   its own 128 features.
3. Apply both to the full batch, and both to a single example `X[:1]`.
4. Set `answers` to a dict with:
   - `"bn_full_std"`, `"ln_full_std"`: the overall std of each full-batch output,
   - `"bn_single_absmax"`: the largest absolute value of batch norm's output on
     the single example (this is what collapse looks like),
   - `"ln_single_std"`: the std of layer norm's output on the single example,
   - `"ln_matches"`: `1.0` if layer norm's output for example 0 is identical
     (to 1e-10) whether computed alone or as part of the full batch, else `0.0`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks batch norm collapses on the single example while layer norm does not, and
that layer norm is batch-independent.

In [ ]:
import numpy as np

rng = np.random.default_rng(2626)
X = rng.normal(2.0, 1.5, (32, 128))
eps = 1e-5
print(f"X {X.shape}, feature 0 across batch: mean {X[:, 0].mean():.3f}, "
      f"std {X[:, 0].std():.3f}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("bn_full_std", [], 0.9999975360112217, -0.16743760224910959, 1e-06),
    ("ln_full_std", [], 0.9999977270395801, -0.1674376342345187, 1e-06),
    ("bn_single_absmax", [], 0.0, 0.0, 1e-06),
    ("ln_single_std", [], 0.9999976477297341, -0.16743762095503553, 1e-06),
    ("ln_matches", [], 1.0, -0.16743801481449916, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "bn_single_absmax" in answers:
    _collapse = answers["bn_single_absmax"] < 1e-6
    _fine = abs(answers["ln_single_std"] - 1.0) < 0.05
    _indep = float(answers["ln_matches"]) == 1.0
    print(f"  {'batchnorm collapses':<24} {'ok' if _collapse else 'wrong'}")
    print(f"  {'layernorm survives':<24} {'ok' if _fine else 'wrong'}")
    print(f"  {'layernorm batch-free':<24} {'ok' if _indep else 'wrong'}")
    _ok = _ok and _collapse and _fine and _indep

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

On a single row the batch variance is exactly zero, so batch norm returns
`0 / sqrt(eps)` for every entry, and the output carries no information at all.
Layer norm's statistics come from the 128 features of that one row, which are
perfectly well defined regardless of batch size, and crucially they do not change
when you add other examples to the batch. That batch-independence is why
transformers use layer norm and why inference with batch size 1 works at all.

```python
import numpy as np

def batchnorm(x):
    m = x.mean(axis=0, keepdims=True)
    v = x.var(axis=0, keepdims=True)
    return (x - m) / np.sqrt(v + eps)

def layernorm(x):
    m = x.mean(axis=1, keepdims=True)
    v = x.var(axis=1, keepdims=True)
    return (x - m) / np.sqrt(v + eps)

bn_full = batchnorm(X)
ln_full = layernorm(X)

single = X[:1]
bn_single = batchnorm(single)
ln_single = layernorm(single)

answers = {
    "bn_full_std": float(bn_full.std()),
    "ln_full_std": float(ln_full.std()),
    "bn_single_absmax": float(np.abs(bn_single).max()),
    "ln_single_std": float(ln_single.std()),
    "ln_matches": float(np.allclose(ln_single, ln_full[:1], atol=1e-10)),
}
```